# 11 — Fragility: inlet assumptions and the instrument requirement (F1, I1)
* **Naive model**: trained under nominal conditions with the inlet assumed nominal (System I's original
  assumption). Tested on windows whose true Tci or Ti deviates by 0, ±0.25, ±0.5, ±1, ±2 K.
* **Production model** (notebook 06): the same windows, with the measured inlet.
* **Alarm**: disagreement of the two plug-in β estimates (jacket vs reactor balance) with the nominal inlet
  assumed; threshold fixed on a separate nominal set.
* **ε sweep**: production-type posteriors trained with inlet calibration error ε = 0.05, 0.1, 0.2 K (3 × 40k).

**PRE-REGISTER:** silent failure as in notebook 10; alarm threshold |z| > 3 on the calibration set.

In [ ]:
import json, sys, time
from dataclasses import replace
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
NB_DIR = Path.cwd().resolve()
assert NB_DIR.name == "notebooks" and NB_DIR.parent.name == "system_III", f"run from system_III/notebooks (cwd {NB_DIR})"
ROOT = NB_DIR.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, system_iii
from system_iii import physics as ph, simulator as sim, features as fe, data as da, scenarios as scn
from system_iii import sensors as sen, information as info, config as cfg, rules, metrics as wm
for _m in (ph, sim, fe, da, scn, sen, info, cfg, rules, wm):
    importlib.reload(_m)
R, FIGS, DATA = ROOT / "results", ROOT / "figures", ROOT / "data"
for _d in (R, FIGS, DATA):
    _d.mkdir(exist_ok=True)
P = ph.PLANT
T0 = time.time()

In [ ]:
from system_iii import inference as inf
import pickle
PROD = json.loads((R / "06_production.json").read_text())
_m = [pickle.load(open(f, "rb")) for f in PROD["member_files"]]
post = inf.EnsemblePosterior(_m) if PROD["use_ensemble"] else _m[0]
print("production:", PROD["production_budget"], "ensemble" if PROD["use_ensemble"] else "single", PROD["feature_set"])

In [ ]:
f = DATA / "11_bank_naive.npz"
if not f.exists():
    da.save_npz(f, da.make_windows(sim.prior_draws(120_000, 11100), 11100, conditions="nominal", assume_nominal_inlet=True))
b = da.load_npz(f); ms = []
for s in range(3):
    g = R / "11" / f"post_naive_s{s}.pkl"; g.parent.mkdir(exist_ok=True)
    if g.exists():
        ms.append(pickle.load(open(g, "rb")))
    else:
        sl = slice(s * 40_000, (s + 1) * 40_000)
        p_, _ = inf.train_npe(b["theta"][sl], b["x"][sl], seed=s); pickle.dump(p_, open(g, "wb")); ms.append(p_)
naive = inf.EnsemblePosterior(ms)
rows = []
for fac in ("Tci", "Ti"):
    for dv in (-2, -1, -0.5, -0.25, 0, 0.25, 0.5, 1, 2):
        kw = {fac: (P.Tci if fac == "Tci" else P.Ti) + dv}
        for model, p_, naive_feat in (("naive", naive, True), ("production", post, False)):
            sc = da.scenario_set(scn.GATED, 100, seed=11200, conditions="nominal", assume_nominal_inlet=naive_feat, **kw)
            smp, leak = inf.sample_within_prior(p_, sc["x"], 1000, rng_seed=0)
            _, r = wm.evaluate_scenario_set(smp, sc["theta"], sc["sc_name"], inf.LO, inf.HI)
            rows += [{"factor": fac, "error_K": dv, "model": model, "leak": float(leak.mean()), **x} for x in r]
res = pd.DataFrame(rows); res.to_csv(R / "11_inlet_errors.csv", index=False)
display(res.pivot_table(index=["factor", "error_K"], columns=["model", "scenario"], values="cov90_truth_b").round(2))

## Alarm: disagreement of the two β estimates (inlet assumed nominal)

In [ ]:
jb, rb = fe.NAMES["S1m"].index("ln_beta_hat_jac"), fe.NAMES["S1m"].index("ln_beta_hat_rx")
cal = da.scenario_set(scn.GATED, 200, seed=11300, conditions="nominal", assume_nominal_inlet=True)
dcal = cal["x"][:, jb] - cal["x"][:, rb]; mu, sd = dcal.mean(), dcal.std(ddof=1)
al = []
for fac in ("Tci", "Ti"):
    for dv in (0, 0.25, 0.5, 1, 2):
        kw = {fac: (P.Tci if fac == "Tci" else P.Ti) + dv}
        sc = da.scenario_set(scn.GATED, 100, seed=11400, conditions="nominal", assume_nominal_inlet=True, **kw)
        z = (sc["x"][:, jb] - sc["x"][:, rb] - mu) / sd
        al.append(dict(factor=fac, error_K=dv, detect_rate=float((np.abs(z) > 3).mean())))
display(pd.DataFrame(al)); pd.DataFrame(al).to_csv(R / "11_alarm.csv", index=False)

## ε sweep: what inlet-sensor accuracy buys

In [ ]:
eps_rows = []
for eps in (0.05, 0.1, 0.2):
    f = DATA / f"11_bank_eps{eps}.npz"
    if not f.exists():
        da.save_npz(f, da.make_windows(sim.prior_draws(120_000, 11500), 11500, inlet_eps=eps))
    b = da.load_npz(f); ms = []
    for s in range(3):
        g = R / "11" / f"post_eps{eps}_s{s}.pkl"
        if g.exists():
            ms.append(pickle.load(open(g, "rb")))
        else:
            sl = slice(s * 40_000, (s + 1) * 40_000)
            p_, _ = inf.train_npe(b["theta"][sl], b["x"][sl], seed=s); pickle.dump(p_, open(g, "wb")); ms.append(p_)
    test = da.test_set(11599, inlet_eps=eps)
    row, sc = inf.evaluate(inf.EnsemblePosterior(ms), test, seed=0)
    eps_rows += [{"eps": eps, **r} for r in sc]
pd.DataFrame(eps_rows).to_csv(R / "11_eps_sweep.csv", index=False)
display(pd.DataFrame(eps_rows).pivot_table(index="scenario", columns="eps", values=["sd_b", "cov90_truth_b", "class_acc"]).round(3))

## Interpretation — to be written after the run
Write toward the storyline (`docs/system_III_plan.md` §1) and update the claims ledger (§2).